In [1]:
import pandas as pd
print(pd.__version__)

2.3.3


In [2]:
import sys
sys.path.append("../src")

from loader import load_season

df = load_season("2024-25")
print(len(df))

737029


In [3]:
df.head()

,clock,actionType,description,playerFullName,teamTricode,gameId,gameDateTimeEst,periodType,period,shortFormattedClock,...,gameType,playerteamId,opponentteamId,jumpBallRecoverdPersonId,gameDateTimeEst_g,gameType_g,gameCategory,clockSec,gameSec,gameDate
0,PT12M00.00S,period,Period Start,None,None,0022400001,2024-11-12 19:00:00,REGULAR,1,None,...,None,None,None,None,None,None,regular,720.0,0.0,2024-11-12 19:00:00
1,PT11M58.00S,jumpball,Jump Ball C. Capela vs. A. Horford: Tip to K. ...,Keaton Wallace,ATL,0022400001,2024-11-12 19:00:00,REGULAR,1,None,...,None,None,None,None,None,None,regular,718.0,2.0,2024-11-12 19:00:00
2,PT11M43.00S,3pt,MISS Z. Risacher 26' 3PT - blocked,Zaccharie Risacher,ATL,0022400001,2024-11-12 19:00:00,REGULAR,1,None,...,None,None,None,None,None,None,regular,703.0,17.0,2024-11-12 19:00:00
3,PT11M43.00S,block,J. Tatum BLOCK (1 BLK),Jayson Tatum,BOS,0022400001,2024-11-12 19:00:00,REGULAR,1,None,...,None,None,None,None,None,None,regular,703.0,17.0,2024-11-12 19:00:00
4,PT11M42.00S,rebound,Z. Risacher REBOUND (Off:1 Def:0),Zaccharie Risacher,ATL,0022400001,2024-11-12 19:00:00,REGULAR,1,None,...,None,None,None,None,None,None,regular,702.0,18.0,2024-11-12 19:00:00


In [4]:
cols = ["period", "clockSec", "actionType", "subType", 
        "shotResult", "teamTricode", "possession"]

df[cols].head(15)

,period,clockSec,actionType,subType,shotResult,teamTricode,possession
0,1,720.0,period,start,None,None,0.0
1,1,718.0,jumpball,recovered,None,ATL,1610612737.0
2,1,703.0,3pt,Jump Shot,Missed,ATL,1610612737.0
3,1,703.0,block,,None,BOS,1610612737.0
4,1,702.0,rebound,offensive,None,ATL,1610612737.0
5,1,698.0,2pt,Jump Shot,Missed,ATL,1610612737.0
6,1,697.0,rebound,defensive,None,BOS,1610612738.0
7,1,684.0,3pt,Jump Shot,Missed,BOS,1610612738.0
8,1,682.0,rebound,defensive,None,ATL,1610612737.0
9,1,677.0,turnover,traveling,None,ATL,1610612737.0


In [5]:
made_fg = df["actionType"].isin(["2pt", "3pt"]) & (df["shotResult"] == "Made")

print(made_fg.head(10))
print("총 야투 성공:", made_fg.sum())

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool
총 야투 성공: 109602


In [6]:
def_reb = (df["actionType"] == "rebound") & (df["subType"] == "defensive")
turnover = df["actionType"] == "turnover"
period_end = (df["actionType"] == "period") & (df["subType"] == "end")

print("수비 리바운드:", def_reb.sum())
print("턴오버:", turnover.sum())
print("쿼터 종료:", period_end.sum())

수비 리바운드: 92981
턴오버: 37693
쿼터 종료: 5350


In [7]:
ft = df[df["actionType"] == "freethrow"]

print(ft["subType"].value_counts())

subType
1 of 2    23787
2 of 2    23776
1 of 1     7341
1 of 3      889
2 of 3      888
3 of 3      887
Name: count, dtype: int64


In [8]:
s = "2 of 2"

print(s.split(" of "))

['2', '2']


In [9]:
s = "2 of 2"
parts = s.split(" of ")

cur = int(parts[0])
total = int(parts[1])

print(cur, total)
print("마지막인가?", cur == total)

2 2
마지막인가? True


In [10]:
s = "1 of 2"
parts = s.split(" of ")

cur = int(parts[0])
total = int(parts[1])

print(cur, total)
print("마지막인가?", cur == total)

1 2
마지막인가? False


In [11]:
def is_last_ft(sub_type):
    if not isinstance(sub_type, str):
        return False
    if " of " not in sub_type:
        return False
    
    cur, total = sub_type.split(" of ")
    return int(cur) == int(total)


print(is_last_ft("2 of 2"))
print(is_last_ft("1 of 2"))
print(is_last_ft(None))

True
False
False


In [12]:
last_ft = (df["actionType"] == "freethrow") & \
          (df["shotResult"] == "Made") & \
          df["subType"].map(is_last_ft)

print("마지막 자유투 성공:", last_ft.sum())

마지막 자유투 성공: 25439


In [13]:
poss_end = made_fg | def_reb | turnover | last_ft | period_end

print("포제션 종료 이벤트:", poss_end.sum())
print("경기당:", round(poss_end.sum() / df["gameId"].nunique(), 1))

포제션 종료 이벤트: 271065
경기당: 205.4


In [14]:
mask = (df["actionType"] == "freethrow") & (df["subType"] == "1 of 1")
idx = df[mask].index[0]

df.loc[idx-3:idx+1, ["clockSec", "actionType", "subType", "shotResult", "teamTricode"]]

,clockSec,actionType,subType,shotResult,teamTricode
95,291.0,freethrow,2 of 2,Made,BOS
96,279.0,2pt,Jump Shot,Made,ATL
97,279.0,foul,personal,None,BOS
98,279.0,freethrow,1 of 1,Made,ATL
99,267.0,foul,personal,None,ATL


In [16]:
made_2back = df["actionType"].shift(2).isin(["2pt", "3pt"]) & \
             (df["shotResult"].shift(2) == "Made")

made_1back = df["actionType"].shift(1).isin(["2pt", "3pt"]) & \
             (df["shotResult"].shift(1) == "Made")

and1 = (df["actionType"] == "freethrow") & (df["subType"] == "1 of 1") & \
       (df["shotResult"] == "Made")

print("1 of 1 성공:", and1.sum())
print("두 칸 앞이 야투성공:", (and1 & made_2back).sum())
print("한 칸 앞이 야투성공:", (and1 & made_1back).sum())

1 of 1 성공: 5642
두 칸 앞이 야투성공: 2849
한 칸 앞이 야투성공: 2


In [17]:
and1_all = (df["actionType"] == "freethrow") & \
           (df["subType"] == "1 of 1") & \
           (made_2back | made_1back)

last_ft_fixed = last_ft & ~and1_all

print("수정 전:", last_ft.sum())
print("수정 후:", last_ft_fixed.sum())
print("제외된 앤드원:", (last_ft & and1_all).sum())

수정 전: 25439
수정 후: 22588
제외된 앤드원: 2851


In [18]:
poss_end_fixed = made_fg | def_reb | turnover | last_ft_fixed | period_end

n_games = df["gameId"].nunique()
per_game = poss_end_fixed.sum() / n_games

print("수정 전 경기당:", round(poss_end.sum() / n_games, 1))
print("수정 후 경기당:", round(per_game, 1))
print("수정 후 팀당  :", round(per_game / 2, 1))
print("리그 실측     : 약 98~100")

수정 전 경기당: 205.4
수정 후 경기당: 203.2
수정 후 팀당  : 101.6
리그 실측     : 약 98~100


In [19]:
tech = df["description"].str.contains("T.FOUL|Technical", case=False, na=False)
print("테크니컬 파울:", tech.sum(), "| 경기당", round(tech.sum()/n_games, 2))

buzzer = made_fg & (df["clockSec"] < 1.0)
print("버저비터 성공:", buzzer.sum(), "| 경기당", round(buzzer.sum()/n_games, 2))

테크니컬 파울: 2522 | 경기당 1.91
버저비터 성공: 597 | 경기당 0.45


In [20]:
tmp = df[["gameId"]].copy()
tmp["end"] = poss_end_fixed

per_game = tmp.groupby("gameId")["end"].sum() / 2

print(per_game.describe().round(1))

count    1320.0
mean      101.6
std         5.4
min        87.0
25%        98.0
50%       101.0
75%       105.0
max       127.5
Name: end, dtype: float64
